# ContextLens — Research Notebook

Investigating impact of retrieval depth K on RAG answer quality.

**K ∈ {1,3,5,10}** | Same docs, chunks, embedding, LLM — only K changes.

This notebook shows:
1. Load processed data
2. Load chunks
3. Load embeddings
4. Load benchmark
5. Run retrieval
6. Generate answers
7. Evaluate
8. Aggregate results
9. Statistical analysis
10. Plot results
11. Inspect failure cases
12. Summarize observations

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from src.config import PAGES_JSONL, CHUNKS_JSONL, BENCHMARK_PATH, RESULTS_DIR, FAISS_INDEX_PATH, EMBEDDINGS_NPY_PATH
from src.utils import load_jsonl, load_json
from src.retriever import get_retriever
from src.generator import get_llm_provider
from src.evaluation import evaluate_run

plt.style.use('default')
sns.set_theme(style="whitegrid")

print(f"Project root: {PROJECT_ROOT}")
print(f"Pages: {PAGES_JSONL.exists()} {PAGES_JSONL}")
print(f"Chunks: {CHUNKS_JSONL.exists()} {CHUNKS_JSONL}")
print(f"Benchmark: {BENCHMARK_PATH.exists()} {BENCHMARK_PATH}")
print(f"FAISS: {FAISS_INDEX_PATH.exists()} {FAISS_INDEX_PATH}")
print(f"Results dir: {RESULTS_DIR}")

## 1. Load processed data

In [ ]:
pages = load_jsonl(PAGES_JSONL) if PAGES_JSONL.exists() else []
print(f"Loaded {len(pages)} pages")
if pages:
    print(f"Sample page: {pages[0]['document']} p.{pages[0]['page']} - {pages[0]['word_count']} words")
    print(pages[0]['text'][:500])

# Stats
import json
stats_path = PROJECT_ROOT / "data" / "processed" / "preprocessing_stats.json"
if stats_path.exists():
    print(json.dumps(load_json(stats_path), indent=2))

## 2. Load chunks

In [ ]:
chunks = load_jsonl(CHUNKS_JSONL) if CHUNKS_JSONL.exists() else []
print(f"Loaded {len(chunks)} chunks")
if chunks:
    df_chunks = pd.DataFrame(chunks)
    print(df_chunks['word_count'].describe())
    print(f"Documents: {df_chunks['document'].unique()}")
    print(f"Sample chunk: {chunks[0]['chunk_id']} - {chunks[0]['word_count']} words")
    print(chunks[0]['text'][:500])

## 3. Load embeddings

In [ ]:
import numpy as np
if EMBEDDINGS_NPY_PATH.exists():
    embs = np.load(str(EMBEDDINGS_NPY_PATH))
    print(f"Embeddings shape: {embs.shape}")
    print(f"Mean norm: {np.linalg.norm(embs, axis=1).mean():.4f}")
else:
    print("Embeddings not found, run python -m src.embeddings")

meta_path = PROJECT_ROOT / "data" / "processed" / "embedding_meta.json"
if meta_path.exists():
    print(json.dumps(load_json(meta_path), indent=2))

## 4. Load benchmark

In [ ]:
benchmark = load_json(BENCHMARK_PATH) if BENCHMARK_PATH.exists() else {"questions":[]}
questions = benchmark.get("questions", []) if isinstance(benchmark, dict) else benchmark
print(f"Loaded {len(questions)} questions")
if questions:
    df_q = pd.DataFrame(questions)
    print(df_q['category'].value_counts())
    print(df_q.head(2)[['question_id','category','question']])

## 5. Run retrieval

In [ ]:
try:
    retriever = get_retriever()
    print(f"Retriever loaded: {retriever.index.ntotal} vectors, type={retriever.model_type}")
    
    # Test query
    test_q = questions[0]['question'] if questions else "What is eligibility for M.Tech?"
    for k in [1,3,5,10]:
        res = retriever.retrieve(test_q, k=k)
        print(f"\nK={k} - Top result: sim={res[0]['similarity']:.4f} doc={res[0]['document']} p.{res[0]['page']}")
        print(res[0]['text'][:300] + "...")
except Exception as e:
    print(f"Retriever failed: {e}")
    print("Run preprocessing, chunking, embeddings first")

## 6. Generate answers

In [ ]:
try:
    llm = get_llm_provider()
    print(f"LLM provider: {llm.provider}, model: {llm.model}")
    
    if questions:
        q = questions[0]
        chunks = retriever.retrieve(q['question'], k=5)
        answer, gen_time, meta = llm.generate(q['question'], chunks)
        print(f"\nQuestion: {q['question']}")
        print(f"Answer ({gen_time:.0f}ms): {answer}")
        print(f"Meta: {meta}")
except Exception as e:
    print(f"Generation failed: {e}")

## 7. Evaluate

In [ ]:
if questions:
    q = questions[0]
    chunks = retriever.retrieve(q['question'], k=3)
    answer, _, _ = llm.generate(q['question'], chunks)
    eval_res = evaluate_run(q['question'], q['ground_truth'], q['required_facts'], answer, chunks)
    print(json.dumps(eval_res, indent=2, default=str)[:2000])

## 8. Aggregate results

In [ ]:
results_csv = RESULTS_DIR / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    print(f"Loaded {len(df)} results")
    print(df.head())
    
    # Aggregate by K
    agg = df.groupby("k").agg({
        "answer_accuracy": ["mean","std"],
        "context_relevance": ["mean","std"],
        "hallucination_rate": ["mean","std"],
        "total_response_time_ms": ["mean","std"]
    }).round(3)
    print(agg)
else:
    print("results.csv not found, run experiment first")
    print("python -m src.experiment --pilot")
    print("python -m src.experiment --full")

## 9. Statistical analysis

In [ ]:
try:
    from scipy.stats import wilcoxon
    import numpy as np
    
    if results_csv.exists():
        df = pd.read_csv(results_csv)
        # Paired test example: accuracy K=1 vs K=3
        pivot = df.pivot(index="question_id", columns="k", values="answer_accuracy")
        if 1 in pivot.columns and 3 in pivot.columns:
            paired = pivot[[1,3]].dropna()
            if len(paired) >= 5:
                stat, p = wilcoxon(paired[1], paired[3])
                print(f"Wilcoxon K=1 vs K=3 accuracy: stat={stat}, p={p}")
                print(f"Mean K=1: {paired[1].mean():.3f}, K=3: {paired[3].mean():.3f}, diff: {paired[3].mean()-paired[1].mean():.3f}")
            else:
                print("Not enough paired data")
except Exception as e:
    print(f"Statistical analysis failed: {e}")

## 10. Plot results

In [ ]:
if results_csv.exists():
    df = pd.read_csv(results_csv)
    
    # Accuracy vs K
    plt.figure(figsize=(8,5))
    acc_by_k = df.groupby("k")["answer_accuracy"].agg(["mean","std"])
    plt.errorbar(acc_by_k.index, acc_by_k["mean"], yerr=acc_by_k["std"], marker='o', capsize=5)
    plt.title("Answer Accuracy vs K")
    plt.xlabel("K")
    plt.ylabel("Accuracy")
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Relevance vs K
    plt.figure(figsize=(8,5))
    rel_by_k = df.groupby("k")["context_relevance"].agg(["mean","std"])
    plt.errorbar(rel_by_k.index, rel_by_k["mean"], yerr=rel_by_k["std"], marker='s', color='green', capsize=5)
    plt.title("Context Relevance vs K")
    plt.xlabel("K")
    plt.ylabel("Relevance")
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Hallucination vs K
    plt.figure(figsize=(8,5))
    hall_by_k = df.groupby("k")["hallucination_rate"].agg(["mean","std"])
    plt.errorbar(hall_by_k.index, hall_by_k["mean"], yerr=hall_by_k["std"], marker='^', color='red', capsize=5)
    plt.title("Hallucination Rate vs K")
    plt.xlabel("K")
    plt.ylabel("Hallucination Rate")
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Response time vs K
    plt.figure(figsize=(8,5))
    time_by_k = df.groupby("k")["total_response_time_ms"].agg(["mean","std"])
    plt.errorbar(time_by_k.index, time_by_k["mean"], yerr=time_by_k["std"], marker='d', color='purple', capsize=5)
    plt.title("Response Time vs K")
    plt.xlabel("K")
    plt.ylabel("Total Time (ms)")
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No results to plot")

## 11. Inspect failure cases

In [ ]:
failure_path = RESULTS_DIR / "failure_cases.json"
if failure_path.exists():
    failures = load_json(failure_path)
    print(f"Found {len(failures)} failure cases")
    for fc in failures[:3]:
        print(f"\n--- {fc['question_id']} | {fc['failure_type']} ---")
        print(f"Q: {fc['question']}")
        print(f"Observed: {fc['observed']}")
        print(f"Explanation: {fc['explanation']}")
else:
    print("No failure cases yet, run experiment")

## 12. Summarize observations

Replace with actual observations from experiment. Do NOT fabricate before running.

In [ ]:
print("""
Observations (to be filled after experiment):
- Accuracy vs K trend: ...
- Relevance vs K trend: ...
- Hallucination vs K trend: ...
- Response time vs K trend: ...
- Statistical significance: ...
- Failure cases insights: ...
- Conclusion: Does more context make AI smarter? ...
""")

# If results exist, auto-summarize
if results_csv.exists():
    df = pd.read_csv(results_csv)
    agg = df.groupby("k").agg({"answer_accuracy":"mean","context_relevance":"mean","hallucination_rate":"mean","total_response_time_ms":"mean"}).round(3)
    print("\nAggregated results:")
    print(agg)
    print("\nExperiment not yet run? Check results/comparison_table.csv")